# Chapter 5 &mdash; LSB-First "Divisible by 3": a Pair-Valued State

**Concept 9 of the Chapter 5 decomposition:** *LSB-First "Divisible by 3": a Pair-Valued State*

Each new bit lands leftmost, so the state must track the running weight $2^k$ as well as the residue.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-LSB-First-Pair-State/Concept-LSB-First-Pair-State.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Reading **least-significant bit first** changes everything: bit $b$ arriving at
position $k$ contributes $b\cdot 2^k$, so the value update is
$N \mapsto N + b\cdot 2^k$ and you need to know $2^k$.

You still cannot store $k$ &mdash; but you can store $2^k \bmod 3$, which cycles
$1, 2, 1, 2, \dots$

So the state is a **pair** $(N\bmod 3,\ 2^k\bmod 3)$: six combinations, of which the
reachable ones form the machine. Same language as the MSB machine, different design.

## 2. Definitions

### The pair-valued recurrence

In [ ]:
def lsb_step(state, b):
    r, w = state                      # (value mod 3, current weight mod 3)
    return ((r + int(b)*w) % 3, (2*w) % 3)

def lsb_resid(s):
    st = (0, 1)
    for b in s: st = lsb_step(st, b)
    return st[0]

### The DFA over pairs, names spelling out the pair

In [ ]:
# Naming: exactly ONE state may begin with 'I'.  Both r=0 states are final,
# so the start state is IF_r0w1 and the other final state is plain F_r0w2.
Div3L = md2mc('''DFA
IF_r0w1 : 0 -> F_r0w2       !! r stays 0, weight 1->2
IF_r0w1 : 1 -> S_r1w2       !! r += 1*1
F_r0w2  : 0 -> IF_r0w1
F_r0w2  : 1 -> S_r2w1       !! r += 1*2
S_r1w2  : 0 -> S_r1w1
S_r1w2  : 1 -> IF_r0w1      !! 1 + 2 = 3 = 0 mod 3
S_r1w1  : 0 -> S_r1w2
S_r1w1  : 1 -> S_r2w2
S_r2w1  : 0 -> S_r2w2
S_r2w1  : 1 -> F_r0w2       !! 2 + 1 = 3 = 0 mod 3
S_r2w2  : 0 -> S_r2w1
S_r2w2  : 1 -> S_r1w1       !! 2 + 2 = 4 = 1 mod 3
''')

### Reference: interpret the string LSB-first

In [ ]:
def val_lsb(s): return int(s[::-1], 2) if s else 0

## 3. Tests

The pair recurrence computes the right residue.

In [ ]:
from itertools import product
bad = [''.join(p) for k in range(1, 12) for p in product('01', repeat=k)
       if lsb_resid(''.join(p)) != val_lsb(''.join(p)) % 3]
print("mismatches :", bad)
assert not bad
print("weight cycle 2^k mod 3 :", [(2**k) % 3 for k in range(8)], " <- period 2")

And the DFA recognises LSB-first divisibility by 3.

In [ ]:
bad = [''.join(p) for k in range(1, 12) for p in product('01', repeat=k)
       if accepts_dfa(Div3L, ''.join(p)) != (val_lsb(''.join(p)) % 3 == 0)]
print("DFA mismatches on 1..11-bit LSB-first numerals :", len(bad))
assert not bad
for s in ['0', '11', '011', '0011']:
    print("  %-6r reads as %-4d divisible by 3? %s"
          % (s, val_lsb(s), accepts_dfa(Div3L, s)))

Six states here versus three for MSB-first &mdash; but minimization tells the real story.

In [ ]:
mL = min_dfa(Div3L)
print("LSB machine: %d states, minimized to %d" % (len(Div3L["Q"]), len(mL["Q"])))
print("\nThe weight component is genuinely needed: reading direction changes the language")
print("of *strings* even though it is the same set of *numbers*.")

## 4. Animation

Follow the pair: the second component alternates every step, the first accumulates.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(Div3L, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Which pairs are unreachable, and why?
2. Build the LSB-first divisible-by-5 machine. What is the weight cycle length?
3. Are the MSB and LSB machines language-equivalent? Check with `langeq_dfa` and explain.

In [ ]:
# Your work for the exercises above.